# Cleaning Dataset Perkebunan — Kabupaten Kediri

Membersihkan satu file: `Produksi_Perkebunan_.csv` (produksi tanaman perkebunan rakyat).

**Hasil pengecekan isi file:**
- Datanya **tingkat Kabupaten saja** (tidak ada kolom kecamatan, tidak ada kolom tahun).
- Kolom nilai berlabel **"ribu ton"**, tapi skalanya mencurigakan (lihat peringatan di cell validasi).
- Nilai `...` di file ini **terdefinisi jelas di catatan kaki** sebagai "Data tidak tersedia", jadi
  diperlakukan sebagai kosong (NaN), bukan 0.
- Di bawah tabel ada baris kosong, baris "Keterangan", dan catatan kaki. Semuanya dibuang dari data.

**Output:** `data/clean/perkebunan/produksi_perkebunan_kabupaten.csv` → kolom `komoditas, produksi, satuan`

In [1]:
# 1. IMPORT LIBRARY
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("✓ Library siap.")

✓ Library siap.


In [2]:
# 2. CARI LOKASI PROJECT (folder 'data') SECARA OTOMATIS
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = None

search_dir = CURRENT_DIR
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    parent_dir = os.path.dirname(search_dir)
    if parent_dir == search_dir:
        break
    search_dir = parent_dir

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder 'data' tidak ditemukan dari direktori notebook ini ke atas. "
        "Pastikan notebook dijalankan dari dalam repository (mis. dari folder 'notebooks/')."
    )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
OUT_DIR = os.path.join(DATA_DIR, "clean", "perkebunan")
os.makedirs(OUT_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output      :", OUT_DIR)

Project root: d:\SDT\Project Analitika Data\AgriDash-0.1
Output      : d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\perkebunan


In [3]:
# 3. CARI FILE SUMBER
# Dicari di data/raw/perkebunan dulu, lalu seluruh data/raw. Nama dicocokkan longgar
# (mis. 'Produksi_Perkebunan_.csv', 'produksi_perkebunan.csv', 'Produksi Perkebunan .csv').

def cari_file_perkebunan():
    kandidat = []
    for root, _, files in os.walk(RAW_DIR):
        for f in files:
            nama = f.lower()
            if nama.endswith(".csv") and "produksi" in nama and "perkebunan" in nama:
                kandidat.append(os.path.join(root, f))
    kandidat.sort(key=lambda p: (os.path.basename(os.path.dirname(p)) != "perkebunan", p))
    return kandidat

kandidat = cari_file_perkebunan()
if not kandidat:
    raise FileNotFoundError(f"File produksi perkebunan tidak ditemukan di {RAW_DIR}")
if len(kandidat) > 1:
    print("⚠️  Ada lebih dari satu kandidat, dipakai yang pertama:", kandidat)

FILE_SUMBER = kandidat[0]
print("File sumber:", FILE_SUMBER)

File sumber: d:\SDT\Project Analitika Data\AgriDash-0.1\data\raw\perkebunan\Produksi Perkebunan .csv


In [4]:
# 4. FUNGSI BANTU

def parse_angka(nilai):
    # '...' = data tidak tersedia (sesuai catatan kaki) -> NaN. '-' -> 0.0 (konvensi BPS).
    if pd.isna(nilai):
        return np.nan
    s = str(nilai).strip()
    if s == "-":
        return 0.0
    if s in ("", "...", "nan"):
        return np.nan
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return np.nan


print("✓ Fungsi bantu siap.")

✓ Fungsi bantu siap.


In [5]:
# 5. BACA FILE & PISAHKAN TABEL DARI CATATAN KAKI

# dtype=str + keep_default_na=False supaya '...' dan sel kosong tidak diubah pandas diam-diam.
df_raw = pd.read_csv(FILE_SUMBER, encoding="utf-8-sig", dtype=str, keep_default_na=False)
kolom_nama, kolom_nilai = df_raw.columns[0], df_raw.columns[1]
print(f"{len(df_raw)} baris terbaca (termasuk baris kosong & catatan kaki).")
print("Kolom:", list(df_raw.columns))

# Tabel berakhir di baris kosong pertama pada kolom nama komoditas; sisanya catatan kaki.
kosong = df_raw[kolom_nama].str.strip().eq("")
posisi_pemisah = int(np.argmax(kosong.values)) if kosong.any() else len(df_raw)
df_data = df_raw.iloc[:posisi_pemisah].copy()
df_catatan = df_raw.iloc[posisi_pemisah:]

teks_catatan = " ".join(df_catatan.astype(str).values.ravel()).strip()
print(f"\nBaris data: {len(df_data)} | baris catatan kaki dibuang: {len(df_catatan)}")
print("Isi catatan kaki:", re.sub(r"\s+", " ", teks_catatan))

if "tidak tersedia" not in teks_catatan.lower():
    print("⚠️  Catatan kaki tidak menyebut arti '...'. Pastikan '...' memang berarti data tidak tersedia.")

14 baris terbaca (termasuk baris kosong & catatan kaki).
Kolom: ['Jenis Tanaman', 'Produksi Tanaman Perkebunan Rakyat (ribu ton) (Ribu Ton)']

Baris data: 11 | baris catatan kaki dibuang: 3
Isi catatan kaki: Keterangan ...) Data tidak tersedia/ Data is not available


In [6]:
# 6. BERSIHKAN: NAMA KOMODITAS, NILAI, SATUAN

# Satuan diambil dari header: '... (ribu ton) (Ribu Ton)' -> 'Ribu Ton'
m = re.match(r"^(.*?)\s*\(([^()]*)\)\s*\(([^()]*)\)$", kolom_nilai.strip())
if m:
    satuan = m.group(3).strip()
else:
    satuan = None
    print(f"⚠️  Satuan tidak bisa dibaca dari header: {kolom_nilai!r}")

# 'Karet/Rubber' -> 'Karet' (ambil nama Indonesia), huruf kapital diseragamkan.
def nama_indonesia(teks):
    return re.sub(r"\s+", " ", teks.split("/")[0]).strip().title()

df_clean = pd.DataFrame({
    "komoditas": df_data[kolom_nama].apply(nama_indonesia),
    "produksi": df_data[kolom_nilai].apply(parse_angka),
    "satuan": satuan,
}).reset_index(drop=True)

display(df_clean)

,komoditas,produksi,satuan
0,Karet,NaN,Ribu Ton
1,Kelapa,684.3,Ribu Ton
2,Kelapa Sawit,NaN,Ribu Ton
3,Kopi,814.9,Ribu Ton
4,Kakao,443.0,Ribu Ton
5,Teh,NaN,Ribu Ton
6,Jambu Mete,56.0,Ribu Ton
7,Pala,NaN,Ribu Ton
8,Cengkeh,494.0,Ribu Ton
9,Gula Tebu,2474198.0,Ribu Ton


In [7]:
# 7. VALIDASI

assert not df_clean["komoditas"].duplicated().any(), "Ada komoditas ganda."
assert (df_clean["komoditas"].str.len() > 0).all()

tidak_tersedia = df_clean.loc[df_clean["produksi"].isna(), "komoditas"].tolist()
print(f"✓ {len(df_clean)} komoditas, {len(df_clean) - len(tidak_tersedia)} punya nilai.")
print(f"  Data tidak tersedia ('...'): {tidak_tersedia}")

# Cek kewajaran skala. Label sumber 'ribu ton', jadi 100 = 100.000 ton per komoditas untuk SATU
# kabupaten. Batas ini hanya pemicu peringatan, bukan koreksi data.
BATAS_WAJAR_RIBU_TON = 100
if satuan and satuan.lower() == "ribu ton":
    mencurigakan = df_clean[df_clean["produksi"] >= BATAS_WAJAR_RIBU_TON]
    if not mencurigakan.empty:
        print("\n⚠️  Skala nilai tidak wajar untuk label 'ribu ton' pada satu kabupaten:")
        display(mencurigakan)
        print("   Contoh: kopi 814,9 ribu ton = 814.900 ton, lebih besar dari produksi kopi satu provinsi.")
        print("   Kemungkinan label satuan salah (mis. sebenarnya ton), atau tiap komoditas satuannya berbeda.")
        print("   Cek ke tabel BPS aslinya sebelum angka ini dipakai. Nilai di output TIDAK diubah.")

✓ 11 komoditas, 7 punya nilai.
  Data tidak tersedia ('...'): ['Karet', 'Kelapa Sawit', 'Teh', 'Pala']

⚠️  Skala nilai tidak wajar untuk label 'ribu ton' pada satu kabupaten:


,komoditas,produksi,satuan
1,Kelapa,684.3,Ribu Ton
3,Kopi,814.9,Ribu Ton
4,Kakao,443.0,Ribu Ton
8,Cengkeh,494.0,Ribu Ton
9,Gula Tebu,2474198.0,Ribu Ton
10,Tembakau,1071.0,Ribu Ton


   Contoh: kopi 814,9 ribu ton = 814.900 ton, lebih besar dari produksi kopi satu provinsi.
   Kemungkinan label satuan salah (mis. sebenarnya ton), atau tiap komoditas satuannya berbeda.
   Cek ke tabel BPS aslinya sebelum angka ini dipakai. Nilai di output TIDAK diubah.


In [8]:
# 8. SIMPAN
out_path = os.path.join(OUT_DIR, "produksi_perkebunan_kabupaten.csv")
df_clean.to_csv(out_path, index=False, encoding="utf-8-sig")

cek = pd.read_csv(out_path, encoding="utf-8-sig")
assert len(cek) == len(df_clean)
print("✓ Tersimpan di:", out_path)
display(cek)

✓ Tersimpan di: d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\perkebunan\produksi_perkebunan_kabupaten.csv


,komoditas,produksi,satuan
0,Karet,NaN,Ribu Ton
1,Kelapa,684.3,Ribu Ton
2,Kelapa Sawit,NaN,Ribu Ton
3,Kopi,814.9,Ribu Ton
4,Kakao,443.0,Ribu Ton
5,Teh,NaN,Ribu Ton
6,Jambu Mete,56.0,Ribu Ton
7,Pala,NaN,Ribu Ton
8,Cengkeh,494.0,Ribu Ton
9,Gula Tebu,2474198.0,Ribu Ton


## Yang perlu kamu cek manual
1. **Satuan.** Label sumber "ribu ton" tidak masuk akal untuk skala satu kabupaten. Nilai disimpan apa adanya
   dan `satuan` mengikuti label sumber, jadi jangan dipakai hitungan (mis. produktivitas, bobot) sebelum
   satuan yang benar dipastikan dari tabel BPS asli.
2. **Gula tebu** (2.474.198) jauh lebih besar dari komoditas lain; kemungkinan satuannya berbeda dari yang lain.
3. **Tidak ada tahun** di file ini. Catat tahun datanya dari sumber, karena dibutuhkan untuk deret waktu.
4. **Hanya tingkat Kabupaten.** Data perkebunan per kecamatan (dan luas areal) belum ada.
5. Komoditas yang `...` (Karet, Kelapa Sawit, Teh, Pala) dibiarkan kosong, bukan 0.